In [89]:
import pandas as pd
import numpy as np

from sklearn.svm import SVC
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import ExtraTreesClassifier

from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_selection import SelectKBest, SelectPercentile, chi2, SelectFpr, SelectFromModel

from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

from sklearn.base import BaseEstimator,TransformerMixin
from sklearn.preprocessing import QuantileTransformer

from sklearn.datasets import make_classification

from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline

In [2]:
rs=12

In [3]:
# german data
def preprocess_german(df, preprocess):
    df['status'] = df['status'].map({'A11': 0, 'A12': 1, 'A13': 2, 'A14': 3}).astype(int)
    df['credit_hist'] = df['credit_hist'].map({'A34': 0, 'A33': 1, 'A32': 2, 'A31': 3, 'A30': 4}).astype(int)

    df['savings'] = df['savings'].map({'A61': 0, 'A62': 1, 'A63': 2, 'A64': 3, 'A65': 4}).astype(int)
    df['employment'] = df['employment'].map({'A71': 0, 'A72': 1, 'A73': 2, 'A74': 3, 'A75': 4}).astype(int)    
    df['gender'] = df['personal_status'].map({'A91': 1, 'A92': 0, 'A93': 1, 'A94': 1, 'A95': 0}).astype(int)
    df['debtors'] = df['debtors'].map({'A101': 0, 'A102': 1, 'A103': 2}).astype(int)
    df['property'] = df['property'].map({'A121': 3, 'A122': 2, 'A123': 1, 'A124': 0}).astype(int)        
    df['install_plans'] = df['install_plans'].map({'A141': 1, 'A142': 1, 'A143': 0}).astype(int)
    if preprocess:
        df = pd.concat([df, pd.get_dummies(df['purpose'], prefix='purpose')],axis=1)
        df = pd.concat([df, pd.get_dummies(df['housing'], prefix='housing')],axis=1)
        df.loc[(df['credit_amt'] <= 2000), 'credit_amt'] = 0
        df.loc[(df['credit_amt'] > 2000) & (df['credit_amt'] <= 5000), 'credit_amt'] = 1
        df.loc[(df['credit_amt'] > 5000), 'credit_amt'] = 2    
        df.loc[(df['duration'] <= 12), 'duration'] = 0
        df.loc[(df['duration'] > 12) & (df['duration'] <= 24), 'duration'] = 1
        df.loc[(df['duration'] > 24) & (df['duration'] <= 36), 'duration'] = 2
        df.loc[(df['duration'] > 36), 'duration'] = 3
        df['age'] = df['age'].apply(lambda x : 1 if x >= 45 else 0) # 1 if old, 0 if young
    df['job'] = df['job'].map({'A171': 0, 'A172': 1, 'A173': 2, 'A174': 3}).astype(int)    
    df['telephone'] = df['telephone'].map({'A191': 0, 'A192': 1}).astype(int)
    df['foreign_worker'] = df['foreign_worker'].map({'A201': 1, 'A202': 0}).astype(int)
    
    df = df.drop(columns=['purpose', 'personal_status', 'housing', 'credit'])

    return df

cols = ['status', 'duration', 'credit_hist', 'purpose', 'credit_amt', 'savings', 'employment',\
            'install_rate', 'personal_status', 'debtors', 'residence', 'property', 'age', 'install_plans',\
            'housing', 'num_credits', 'job', 'num_liable', 'telephone', 'foreign_worker', 'credit']
df = pd.read_table('german.data', names=cols, sep=" ", index_col=False)
df['credit'] = df['credit'].replace(2, 0) #1 = Good, 2= Bad credit risk
y = df['credit']
df = preprocess_german(df, True)

X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2, random_state=rs)

In [37]:
# adult data
def process_adult(df):
    # replace missing values (?) to nan and then drop the columns
    df['country'] = df['country'].replace('?',np.nan)
    df['workclass'] = df['workclass'].replace('?',np.nan)
    df['occupation'] = df['occupation'].replace('?',np.nan)
    # dropping the NaN rows now
#     df.dropna(how='any',inplace=True)
    df['income'] = df['income'].map({'<=50K': 0, '>50K': 1}).astype(int)
    df['age'] = df['age'].apply(lambda x : 1 if x >= 45 else 0) # 1 if old, 0 if young
    df['workclass'] = df['workclass'].map({'Never-worked': 0, 'Without-pay': 1, 'State-gov': 2, 'Local-gov': 3, 'Federal-gov': 4, 'Self-emp-inc': 5, 'Self-emp-not-inc': 6, 'Private': 7})
    df['education'] = df['education'].map({'Preschool': 0, '1st-4th': 1, '5th-6th': 2, '7th-8th': 3, '9th': 4, '10th': 5, '11th': 6, '12th': 7, 'HS-grad':8, 'Some-college': 9, 'Bachelors': 10, 'Prof-school': 11, 'Assoc-acdm': 12, 'Assoc-voc': 13, 'Masters': 14, 'Doctorate': 15}).astype(int)
    df['marital'] = df['marital'].map({'Married-civ-spouse': 2, 'Divorced': 1, 'Never-married': 0, 'Separated': 1, 'Widowed': 1, 'Married-spouse-absent': 2, 'Married-AF-spouse': 2}).astype(int)
    df['relationship'] = df['relationship'].map({'Wife': 1 , 'Own-child': 0 , 'Husband': 1, 'Not-in-family': 0, 'Other-relative': 0, 'Unmarried': 0}).astype(int)
    df['race'] = df['race'].map({'White': 1, 'Asian-Pac-Islander': 0, 'Amer-Indian-Eskimo': 0, 'Other': 0, 'Black': 0}).astype(int)
    df['gender'] = df['gender'].map({'Male': 1, 'Female': 0}).astype(int)
    # process hours
    df.loc[(df['hours'] <= 40), 'hours'] = 0
    df.loc[(df['hours'] > 40), 'hours'] = 1
    df = df.drop(columns=['fnlwgt', 'education.num', 'occupation', 'country', 'capgain', 'caploss'])
    df = df.reset_index(drop=True)
    return df

cols = ['age', 'workclass', 'fnlwgt', 'education', 'education.num', 'marital', 'occupation',\
            'relationship', 'race', 'gender', 'capgain', 'caploss', 'hours', 'country', 'income']

df_train = pd.read_csv('adult.data', names=cols, sep=",")
df_test = pd.read_csv('adult.test', names=cols, sep=",")

df_train = process_adult(df_train)
df_test = process_adult(df_test)

df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
X_train = df_train.drop(columns='income')
y_train = df_train['income']

X_test = df_test.drop(columns='income')
y_test = df_test['income']

In [9]:
# X, y = make_classification(random_state=rs)
# X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=rs)

In [88]:
# Regression
transformer = QuantileTransformer(output_distribution='normal')
regressor = LinearRegression()
regr = TransformedTargetRegressor(regressor=regressor,
                                  transformer=transformer)
pipe = Pipeline([('imputer', KNNImputer(n_neighbors=50)), ('scaler', StandardScaler()), ('linear_reg', regr)])
pipe.fit(X_train, y_train) # should choose regression datasets
pipe.score(X_test, y_test)

-0.12988244017094863

In [49]:
# All intermediate steps should be transformers and implement fit and transform

pipe = Pipeline([('imputer', KNNImputer(n_neighbors=50)), ('scaler', StandardScaler()), ('svc', SVC())])
pipe.fit(X_train, y_train)
pipe.score(X_test, y_test)

0.8162275044530434

In [50]:
pipe = Pipeline([('imputer', KNNImputer(n_neighbors=50)), ('scaler', StandardScaler()), ('lr', LogisticRegression())])
pipe.fit(X_train, y_train)
pipe.score(X_test, y_test)

0.8089797923960445

In [52]:
pipe = Pipeline([('imputer', KNNImputer(n_neighbors=50)), ('scaler', RobustScaler()), ('lr', LogisticRegression())])
pipe.fit(X_train, y_train)
pipe.score(X_test, y_test)

0.8089797923960445

In [53]:
pipe = Pipeline([('imputer', KNNImputer(n_neighbors=50)), ('scaler', MinMaxScaler()), ('lr', LogisticRegression())])
pipe.fit(X_train, y_train)
pipe.score(X_test, y_test)

0.8088569498188072

In [55]:
pipe = Pipeline([('imputer', KNNImputer(n_neighbors=50)), ('feature_selector', SelectKBest(chi2, k=7)), ('lr', LogisticRegression())])
pipe.fit(X_train, y_train)
pipe.score(X_test, y_test)

0.8118665929611203

In [78]:
pipe = Pipeline([('imputer', KNNImputer(n_neighbors=50)), ('feature_selector', SelectPercentile(chi2, percentile=10)), ('lr', LogisticRegression())])
pipe.fit(X_train, y_train)
pipe.score(X_test, y_test)

0.7637737239727289

In [56]:
pipe = Pipeline([('imputer', KNNImputer(n_neighbors=50)), ('feature_selector', SelectFpr(chi2, alpha=0.01)), ('lr', LogisticRegression())])
pipe.fit(X_train, y_train)
pipe.score(X_test, y_test)

0.8089797923960445

In [79]:
pipe = Pipeline([('imputer', KNNImputer(n_neighbors=50)), ('lsvc', SelectFromModel(LinearSVC(C=0.01, penalty="l1", dual=False))), ('lr', LogisticRegression())])
pipe.fit(X_train, y_train)
pipe.score(X_test, y_test)

0.8089797923960445

In [58]:
pipe = Pipeline([('imputer', KNNImputer(n_neighbors=50)), ('scaler', MinMaxScaler()), ('feature_selector', SelectKBest(chi2, k=7)), ('lr', LogisticRegression())])
pipe.fit(X_train, y_train)
pipe.score(X_test, y_test)

0.8091026349732817

In [59]:
pipe = Pipeline([('imputer', KNNImputer(n_neighbors=50)), ('scaler', MinMaxScaler()), ('lr', LogisticRegression())])
pipe.fit(X_train, y_train)
pipe.score(X_test, y_test)

0.8088569498188072

In [60]:
pipe = Pipeline([('imputer', KNNImputer(n_neighbors=50)), ('scaler', MinMaxScaler()), ('feature_selector', SelectKBest(chi2, k=7)), ('lr', LogisticRegression())])
pipe.fit(X_train, y_train)
pipe.score(X_test, y_test)

0.8091026349732817

In [62]:
# iterative imputer for Multiple Imputation by Chained Equations (MICE)
pipe = Pipeline([('imputer', IterativeImputer(random_state=0)), ('scaler', MinMaxScaler()), ('feature_selector', SelectKBest(chi2, k=7)), ('lr', LogisticRegression())])
pipe.fit(X_train, y_train)
pipe.score(X_test, y_test)

0.8091026349732817

In [63]:
# https://towardsdatascience.com/creating-custom-transformers-using-scikit-learn-5f9db7d7fdb5
# (masquerading outlier detector as a transformer)

# Outlier detection using IQR
class OutlierRemover(BaseEstimator,TransformerMixin):
    def __init__(self,factor=1.5):
        self.factor = factor
        
    def outlier_detector(self,X,y=None):
        X = pd.Series(X).copy()
        q1 = X.quantile(0.25)
        q3 = X.quantile(0.75)
        iqr = q3 - q1
        self.lower_bound.append(q1 - (self.factor * iqr))
        self.upper_bound.append(q3 + (self.factor * iqr))

    def fit(self,X,y=None):
        self.lower_bound = []
        self.upper_bound = []
        X.apply(self.outlier_detector)
        return self
    
    def transform(self,X,y=None):
        X = pd.DataFrame(X).copy()
        for i in range(X.shape[1]):
            x = X.iloc[:, i].copy()
            x[(x < self.lower_bound[i]) | (x > self.upper_bound[i])] = np.nan
            X.iloc[:, i] = x
        return X
    
outlier_remover = OutlierRemover()

In [70]:
pipe = Pipeline([('outlier', OutlierRemover()), ('imputer', KNNImputer(n_neighbors=10)), ('lr', LogisticRegression())])
pipe.fit(X_train, y_train)
pipe.score(X_test, y_test)

0.7977396965788343

In [71]:
pipe = Pipeline([('outlier', OutlierRemover()), ('imputer', KNNImputer(n_neighbors=50)), ('lr', LogisticRegression())])
pipe.fit(X_train, y_train)
pipe.score(X_test, y_test)

/Users/rpradhan/Library/Python/3.8/lib/python/site-packages/sklearn/linear_model/_logistic.py:814: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.7986610159081138

**GRASP implementation (Greedy Randomized Adaptive Search Procedure (GRASP) meta-heuristic)**

In each of the r iterations:

    # Phase 1: Construction of feasible solutions
    For the given pipeline, generate a solution: a set of parameters for the pipeline

    # Phase 2: Local search for the local optimum
    For the generated solution, do a local search to get the best set of parameters (local optimum)
    
Collect all the local optima, and return the best solution

In [ ]:
def ConstructSolution():
    

def LocalSearch():